In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# ניהול ביצועים: פונקציית Qiskit מאת Q-CTRL Fire Opal
*ראו את [ה-API reference](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי תוכנית IBM Quantum&reg; Premium Plan, Flex Plan ו-On-Prem (דרך IBM Quantum Platform API). הן בסטטוס גרסת תצוגה מקדימה וכפופות לשינויים.


<Accordion>
<AccordionItem title="Package versions">

הקוד בעמוד זה פותח באמצעות הדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או חדשות יותר.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## סקירה כללית
Fire Opal Performance Management מאפשרת לכולם להשיג תוצאות משמעותיות ממחשבי קוונטום בקנה מידה גדול, מבלי שיצטרכו להיות מומחי חומרה קוונטית. בעת הרצת Circuit-ים עם Fire Opal Performance Management, טכניקות דיכוי שגיאות מבוססות AI מיושמות אוטומטית, ומאפשרות לדרג בעיות גדולות יותר עם יותר Gate-ים ו-Qubit-ים. גישה זו מצמצמת את מספר ה-shots הנדרשים להגעה לתשובה הנכונה, ללא תקורה נוספת — מה שחוסך זמן חישוב ועלויות משמעותיות.

ניהול ביצועים מדכא שגיאות ומגדיל את ההסתברות לקבל את התשובה הנכונה על חומרה רועשת. במילים אחרות, הוא מגדיל את יחס האות-לרעש. התמונה הבאה מציגה כיצד הדיוק המשופר שמאפשר ניהול ביצועים יכול לצמצם את הצורך ב-shots נוספים במקרה של אלגוריתם Quantum Fourier Transform עם 10 qubit-ים. עם 30 shots בלבד, Q-CTRL מגיעה לסף הביטחון של 99%, בעוד ש-Sampler ברירת המחדל (‏`QiskitRuntime` Sampler, ‏`optimization_level`=3 ו-`resilience_level`=1, ‏`ibm_sherbrooke`) דורש 170,000 shots. בכך שמגיעים לתשובה הנכונה מהר יותר, חוסכים זמן ריצה חישובי משמעותי.

![ויזואליזציה של זמן הריצה המשופר](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

אפשר להשתמש בפונקציית ניהול הביצועים עם כל אלגוריתם, וניתן לשלב אותה בקלות במקום ה-[Qiskit Runtime primitives](/guides/primitives) הסטנדרטיים. מאחורי הקלעים, מספר טכניקות דיכוי שגיאות פועלות יחד כדי למנוע שגיאות בזמן ריצה. כל שיטות ה-pipeline של Fire Opal מוגדרות מראש ואינן תלויות באלגוריתם, כך שתמיד מקבלים את הביצועים הטובים ביותר ישר מהקופסה.

לקבלת גישה לניהול ביצועים, [צרו קשר עם Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## תיאור
ל-Fire Opal Performance Management שתי אפשרויות ריצה הדומות ל-Qiskit Runtime primitives, כך שניתן להחליף בקלות את Q-CTRL Sampler ו-Estimator. זרימת העבודה הכללית לשימוש בפונקציית ניהול הביצועים היא:
1. הגדירו את ה-Circuit (ואת האופרטורים במקרה של ה-Estimator).
2. הריצו את ה-Circuit.
3. אחזרו את התוצאות.

כדי לצמצם את הרעש של החומרה, Fire Opal משתמשת במגוון טכניקות דיכוי שגיאות מבוססות AI כפי שמוצג בתמונה הבאה. עם Fire Opal, כל ה-pipeline אוטומטי לחלוטין וללא כל צורך בהגדרות.

ה-pipeline של Fire Opal מבטל את הצורך בתקורה נוספת, כמו זמן ריצה קוונטי מוגדל או qubit-ים פיזיים נוספים. שימו לב שזמן עיבוד קלאסי עדיין גורם לעלויות (ראו את הקטע [מדדי ביצועים](#benchmarks) לאומדנים, כאשר "Total time" משקף הן עיבוד קלאסי והן קוונטי). בניגוד לתיקון שגיאות, שדורש תקורה בצורת דגימה, דיכוי השגיאות של Fire Opal פועל הן ברמת ה-Gate והן ברמת הפולס כדי לטפל במקורות שונים של רעש ולמנוע את ההסתברות להתרחשות שגיאה. בכך שמונעים שגיאות, מבטלים את הצורך בעיבוד לאחר המדידה יקר.

התמונה הבאה מציגה את שיטות דיכוי השגיאות שמאפשר Fire Opal Performance Management.

![ויזואליזציה של pipeline דיכוי השגיאות](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

הפונקציה מציעה שני primitives, Sampler ו-Estimator, והקלט והפלט של שניהם מרחיבים את המפרט המיושם עבור [Qiskit Runtime V2 primitives](/guides/primitive-input-output#pubs).
## מדדי ביצועים
תוצאות [בנצ'מרקינג אלגוריתמי שפורסמו](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) מדגימות שיפור ביצועים משמעותי על פני אלגוריתמים שונים, כולל Bernstein-Vazirani, Quantum Fourier Transform, חיפוש Grover, אלגוריתם אופטימיזציה קוונטית מקורבת, ו-Variational Quantum Eigensolver. שאר החלק הזה מספק פרטים נוספים על סוגי האלגוריתמים שניתן להריץ, כמו גם את הביצועים וזמני הריצה הצפויים.

המחקרים העצמאיים הבאים מדגימים כיצד ניהול הביצועים של Q-CTRL מאפשר מחקר אלגוריתמי בקנה מידה שובר שיאים:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) - למידת גרעין קוונטי עד 50 qubit-ים
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) - אומדן פאזה קוונטית עד 33 qubit-ים
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) - טעינת נתונים קוונטית עד 21 qubit-ים

הטבלה הבאה מספקת מדריך גס לדיוק ולזמני ריצה מריצות בנצ'מרקינג קודמות על `ibm_fez`. הביצועים על מכשירים אחרים עשויים להשתנות. זמן השימוש מבוסס על הנחה של 10,000 shots לכל Circuit. "מספר ה-Qubit-ים" המצוין אינו מגבלה קשה אלא מייצג סף גס שבו ניתן לצפות לדיוק עקבי מאוד בפתרונות. גדלי בעיות גדולים יותר נפתרו בהצלחה, ובדיקה מעבר למגבלות אלה מוזמנת.

| דוגמה    | מספר qubit-ים | דיוק | מדד דיוק | זמן כולל (שניות) | שימוש ב-Runtime (שניות) | Primitive (מצב) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | שיעור הצלחה (אחוז הריצות שבהן התשובה הנכונה היא ה-bitstring בעל הספירה הגבוהה ביותר)     | 10    | 8         | Sampler |
| Quantum Fourier Transform | 30Q              | 100% | שיעור הצלחה (אחוז הריצות שבהן התשובה הנכונה היא ה-bitstring בעל הספירה הגבוהה ביותר)      | 10    | 8        | Sampler |
| Quantum Phase Estimation  | 30Q   | 99.9998%  | דיוק הזווית שנמצאה: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| סימולציה קוונטית: מודל Ising (15 צעדים) | 20Q  | 99.775%   |  $A$ (מוגדר למטה)  |  60 (לכל צעד)  | 15 (לכל צעד)   | Estimator |
| סימולציה קוונטית 2: דינמיקה מולקולרית (20 נקודות זמן) | 34Q  |  96.78%  |  $A_{mean}$ (מוגדר למטה)   | 10 (לכל נקודת זמן)    | 6 (לכל נקודת זמן)  | Estimator |

הגדרת דיוק מדידת ערך ציפייה — המדד $A$ מוגדר כדלקמן:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
כאשר $ \epsilon^{ideal} $ = ערך ציפייה אידיאלי, $ \epsilon^{meas} $ = ערך ציפייה נמדד, $\epsilon^{ideal}_{max} $ = ערך מקסימלי אידיאלי, ו-$\epsilon^{ideal}_{min}$ = ערך מינימלי אידיאלי. $A_{mean}$ הוא פשוט הממוצע של ערך $A$ על פני מדידות מרובות.

מדד זה משמש מפני שהוא אינווריאנטי להזזות גלובליות ולסקלה בטווח הערכים האפשריים. במילים אחרות, בין אם תגבילו את טווח ערכי הציפייה האפשריים למעלה או למטה ובין אם תגדילו את הפריסה, ערך $A$ אמור להישאר עקבי.
## תחילת עבודה
Fire Opal Performance Management משתמשת ב-Qiskit גרסה `2.0.0`, שהיא הגרסה המומלצת. הגרסאות הנתמכות הן Qiskit >=v`2.0.0`.
אמתו את הזהות שלכם באמצעות [מפתח ה-API של IBM Quantum Platform](http://quantum.cloud.ibm.com/), ובחרו את פונקציית ה-Qiskit כך. (הקטע הזה מניח שכבר [שמרתם את החשבון שלכם](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלכם.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. הרצת ה-Circuit**

הריצו את ה-Circuit, ובאופן אופציונלי הגדירו את ה-Backend ואת מספר ה-shots.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

ניתן להשתמש ב-[Qiskit Serverless APIs](/guides/serverless) המוכרים לבדיקת סטטוס עומס העבודה של פונקציית ה-Qiskit:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. אחזור התוצאה**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

התוצאות הן באותו פורמט כמו [תוצאת Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Sampler primitive
### דוגמה של Sampler
השתמש ב-Sampler primitive של Fire Opal Performance Management כדי להריץ מעגל Bernstein–Vazirani. האלגוריתם הזה, שמשמש למציאת מחרוזת נסתרת מהפלטים של פונקציית קופסה שחורה, הוא אלגוריתם בנצ'מרק נפוץ כי יש לו תשובה נכונה אחת בלבד.
**1. יצירת ה-Circuit**

הגדר את התשובה הנכונה לאלגוריתם, את המחרוזת הנסתרת, ואת ה-Circuit של Bernstein–Vazirani. אפשר לשנות את רוחב ה-Circuit פשוט על ידי שינוי `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. הרצת ה-Circuit**

הרץ את ה-Circuit ואפשר להגדיר את ה-Backend ואת מספר ה-shots.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

בדוק את [הסטטוס](/guides/functions#check-job-status) של עבודת Qiskit Function שלך או קבל [תוצאות](/guides/functions#retrieve-results) כך:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. שרטוט המחרוזות הבינאריות המובילות**

שרטט את המחרוזת הבינארית עם הכמות הגבוהה ביותר כדי לראות אם המחרוזת הנסתרת היא ה-mode.